In [ ]:
import pandas as pd
from astropy.coordinates.angles.formats import hours_to_string
from dask.array import average
from gototile.grid import SkyGrid
from goto_sim.scheduling import HEATSurvey
from goto_sim.utils import load_tilelist
from astropy.time import Time, TimeDelta
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib import colormaps
import glob
import matplotlib.font_manager

plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
data_dir = "sims"
figure_dir = "figures"
file_pattern = f"{data_dir}/simulation_results_reserve_0.*.csv"
file_list = glob.glob(file_pattern)
if not file_list:
    raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
print(f"Found {len(file_list)} files matching the pattern.")
reserve_fractions = [float(x .split('_')[-1].split('.csv')[0]) for x in file_list]
results = {}
for file_path, reserve_fraction in zip(file_list, reserve_fractions):
    df = pd.read_csv(file_path, comment='#')
    results[reserve_fraction] = df
results_df = pd.concat(results, names=['reserve_fraction', 'index']).reset_index(level='reserve_fraction')

heats_survey = HEATSurvey()
heats_tiles = heats_survey.tiles
cold_survey = heats_survey.generate_colds()
cold_tiles = cold_survey.get_tiles()
grid: SkyGrid = SkyGrid.from_name("GOTO")

target_dict = {}
for tile in grid.tilenames:
    if tile in heats_tiles:
        target_dict[tile] = 'HEATS'
    elif tile in cold_tiles:
        target_dict[tile] = 'COLDs'
    else:
        target_dict[tile] = 'Other'
target_dict['ToO'] = 'ToO'
target_dict['Reserved'] = 'Reserved'

In [ ]:
allsky_file_pattern = f"{data_dir}/simulation_results_reserve_allsky_*.csv"
allsky_file_list = glob.glob(allsky_file_pattern)
print(f"Found {len(allsky_file_list)} allsky files matching the pattern.")
results_allsky = {}
for file_path, reserve_fraction in zip(allsky_file_list, reserve_fractions):
    df = pd.read_csv(file_path, comment='#')
    results_allsky[reserve_fraction] = df
results_allsky_df = pd.concat(results_allsky, names=['reserve_fraction', 'index']).reset_index(level='reserve_fraction')
results_allsky_df

In [ ]:
results_df['target'] = results_df['target'].map(lambda x: target_dict.get(x,x))
results_df['pandas_time'] = pd.to_datetime(results_df['time'] + 2400000.5, unit='D', origin='julian')
results_df['diff'] = results_df.groupby(['tile', 'reserve_fraction'])['time'].diff()
results_df

In [ ]:
# Replace HEATS target tiles with HEATS, COLDs tiles with COLDs.
# This is fairly slow, probably a faster way to do this with mapping
#results_df['target'] = results_df['target'].replace({tile: 'HEATS' for tile in heats_tiles})
#results_df['target'] = results_df['target'].replace({tile: 'COLDs' for tile in cold_tiles})
#results_df

In [ ]:
cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.median(np.diff(np.sort(x)))
).sort_index()

cadence_allsky = results_allsky_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.median(np.diff(np.sort(x)))
).sort_index()

print(f"Total results: {len(cadence)}")
cold_cadences = cadence.loc[cold_tiles]
print(f"COLD results: {len(cold_cadences)}")

heats_cadences = cadence.loc[heats_tiles]
print(f"HEATS results: {len(heats_cadences)}")
n_observations = results_df.groupby("reserve_fraction").aggregate("count").loc[0]["time"]
total_time_seconds = n_observations * 5 * 60
total_time_hours = total_time_seconds / 3600
print(f"Total Observations: {n_observations}\n Total Time: {total_time_hours} hours")

In [ ]:
average_cold_cadence = cold_cadences.groupby('reserve_fraction').mean()
average_heats_cadence = heats_cadences.groupby('reserve_fraction').mean()
average_allsky_cadence = cadence.groupby('reserve_fraction').mean()
# Print a table of the results for COLD survey
print("COLD Survey Average Cadence:")
print("Reserve Fraction | Average Cadence | Reserve Fraction ")
print(" ----------------|-----------------|------------------")
for i, row in average_cold_cadence.items():
    print(f"{i/2:.2f}             | {row:.02f}            | {1 - i/2:.2f}")

### Pretty Version of Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='s', color='tab:red')
ax.plot(average_allsky_cadence.index/2, average_allsky_cadence.values, label='Comparison\nAll-sky Survey', marker="^", color='tab:grey', linestyle='dashed', alpha=0.9)
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserved Time')

# Plot fraction of array time for COLDS survey
#ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
# Plot fraction of array time as number of observations per year
hours_axis = ax.secondary_xaxis('top', functions=(lambda x: x * total_time_hours, lambda x: x / total_time_hours))
hours_axis.set_xlabel('Hours Reserved')
hours_axis.xaxis.set_minor_locator(plt.MultipleLocator(200))
hours_axis.tick_params(direction="inout", which="minor", length=3)
if True:
    pointing_axis = ax.secondary_xaxis(1, functions=(lambda x: x * total_time_hours * 60 / 5, lambda x: x / total_time_hours * 60 / 5))
    pointing_axis.set_xlabel('Pointings Reserved', labelpad=-27)
    # Make ticks point inwards
    pointing_axis.tick_params(axis="x", direction="in", pad=-15)
    pointing_axis.xaxis.set_ticks([x*12 for x in [0, 1000, 2000, 3000, 4000, 5000, 6000]])

ax.legend()

ax.set_xlim(-0.01,0.46)
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.025))

#ax.set_yscale("log")
ax.yaxis.set_major_locator(plt.MultipleLocator(5))
ax.yaxis.set_major_formatter(plt.ScalarFormatter())
ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
plt.ylim(0, 27)
plt.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 0.95))
plt.grid(alpha=0.7, linewidth=0.5)
plt.grid(which='minor', alpha=0.2, linestyle='dashed')
plt.savefig(f"{figure_dir}/reservefraction_vs_cadence.png", dpi=300)
plt.show()

In [ ]:
# Web version without excess spines and grid

fig, ax = plt.subplots(figsize=(10, 6))
ax.spines[['right', 'top']].set_visible(False)
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label=f'COLD Survey: {len(cold_tiles)} Tiles', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label=f'HEATS: {len(heats_tiles)} Tiles', marker='s', color='tab:red')
ax.plot(average_allsky_cadence.index/2, average_allsky_cadence.values, label = f"All-sky Survey\n(For Comparison)",marker="^", color='silver')
ax.set_xlabel('Fraction of Array Time Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Effect of Reserved Fraction on Survey Cadence')

# Plot fraction of array time for COLDS survey
#ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
# Plot fraction of array time as number of observations per year
ax.secondary_xaxis('top', functions=(lambda x: x * total_time_hours, lambda x: x / total_time_hours)).set_xlabel('Hours Reserved')
ax.legend()
if True:
    pointing_axis = ax.secondary_xaxis(1, functions=(lambda x: x * total_time_hours * 60 / 5, lambda x: x / total_time_hours * 60 / 5))
    pointing_axis.set_xlabel('Pointings Reserved', labelpad=-27)
    # Make ticks point inwards
    pointing_axis.tick_params(axis="x", direction="in", pad=-15)
    pointing_axis.xaxis.set_ticks([x*12 for x in [0, 1000, 2000, 3000, 4000, 5000, 6000]])

# Add text labels
ax.text(0.465, 1.3, "HEATS\nTiles", color='tab:red', fontsize=12, ha='center', va='center')
ax.text(0.470, 17.9, "All-sky\nSurvey", color='silver', fontsize=12, ha='center', va='center')
ax.text(0.469, 24, "COLD\nTiles", color='tab:blue', fontsize=12, ha='center', va='center')

ax.set_xlim(-0.005,0.46)
ax.xaxis.set_major_locator(plt.MultipleLocator(0.05))

#ax.set_yscale("log")
ax.yaxis.set_major_locator(plt.MultipleLocator(3))
ax.yaxis.set_major_formatter(plt.ScalarFormatter())
ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
plt.ylim(0, 25.5)
plt.grid(alpha=0.4, linestyle='dashed', linewidth=0.5)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{figure_dir}/reservefraction_vs_cadence_simple.png", dpi=300)
plt.show()

In [ ]:
# Bar chart of number of observations per target for each reserve fraction
observations_per_target = results_df.groupby(['reserve_fraction', 'target']).size().unstack(fill_value=0)
observations_per_target = observations_per_target.iloc[:, [1, 3, 2, 0]]
observations_per_target

In [ ]:
fraction_per_target = observations_per_target.div(observations_per_target.sum(axis=1), axis=0)
fraction_per_target
array_fraction_per_target = fraction_per_target.copy()
array_fraction_per_target.index = array_fraction_per_target.index / 2

In [ ]:
ax = array_fraction_per_target.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=['tab:red', 'tab:grey', 'tab:green', 'tab:blue'],
    alpha=1
)
ax.set_xlabel('Fraction of Array Time Reserved for Other Surveys')
ax.set_ylabel('Fraction of Observations')
ax.set_title('Fraction of Observations per Target vs Reserve Fraction')
for container in ax.containers:
    ax.bar_label(container, fmt='{:.0%}', label_type='center', fontsize=5, color='white')
plt.legend(title='Target', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{figure_dir}/array_fraction_barchart.png", dpi=300)
plt.show()

In [ ]:
observations_per_tile = results_df.groupby(['reserve_fraction', 'tile']).size()
observations_per_tile = observations_per_tile.reindex(grid.tilenames, level='tile', fill_value=0)
observations_per_tile

In [ ]:
cadence.loc[heats_tiles,0.00]

In [ ]:
for (reserve_fraction, group), (reserve_fraction, observations) in zip(cadence.groupby('reserve_fraction'), observations_per_tile.groupby('reserve_fraction')):
    if reserve_fraction == 0:
        print(len(observations.values))
        print(group.values)


In [ ]:
# for each reserve fraction, plot the cadence and total number of observations for each tile
for (reserve_fraction, group_cadence) in cadence.groupby('reserve_fraction'):
    group_observations = observations_per_tile.loc[reserve_fraction]
    heats_observations = group_observations.loc[heats_tiles]
    cold_observations = group_observations.loc[cold_tiles]
    print(f"Reserve Fraction: {reserve_fraction/2:.2f}")
    if True:
        fig = plt.figure(figsize=(12, 8), dpi=100)
        fig.suptitle(f'Reserve Fraction: {reserve_fraction/2:.2f}')
        gs = GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

        # -------------------------
        axes = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
        axes.grid()

        tiles = grid.plot_tiles(axes, array=group_observations.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)
        count_ticks = np.arange(round(min(group_observations.values)-10, -1) if min(group_observations)> 20 else 0, max(group_observations.values) + 1, round(max(group_observations.values)//5, -1) if max(group_observations.values) > 100 else 2)
        fig.colorbar(
            tiles, ax=axes, ticks=count_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label=f'Total number of observations'
            f'\n(min={min(group_observations.values):.0f}, '
            f'mean={np.mean(group_observations.values):.1f}, '
            f'max={max(group_observations.values):.0f})'
        )
        # ------------------------
        axes = fig.add_subplot(gs[1, 0], projection='astro hours mollweide')
        axes.grid()
        tiles = grid.plot_tiles(axes, array=group_cadence.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=np.array(grid.tilenames)[np.array(group_cadence.values) == 0],
                        fc='0.4', ec='none', zorder=0.7)

        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)

        cadence_ticks = np.arange(0, int(max(group_cadence.values))+2, (max(group_cadence.values)//5 if max(group_cadence.values) > 5 else 1))
        fig.colorbar(
            tiles, ax=axes, ticks=cadence_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label='Median days between observations'
            f'\n(min={min(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'mean={np.mean(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'max={max(group_cadence.values):.1f})'
        )
        # -------------------------
        axes = fig.add_subplot(gs[0, 1])
        counts, bins, patches = axes.hist([heats_observations, cold_observations],
                bins=np.arange(0, int(max(group_observations.values))+1, 10), stacked=True,
                color=['red', 'blue'],
                label=['Tiles in the HEATS survey (n={})'.format(len(heats_tiles)),
                       'Tiles in the COLD survey (n={})'.format(len(cold_tiles))],
        )
        axes.legend(loc='upper right', bbox_to_anchor=(1, 1.25), frameon=False)
        axes.set_xlabel('Number of observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(int(min(group_observations.values))-1, int(max(group_observations.values))+1+0.5)
        axes.set_xticks(count_ticks)
        # -------------------------
        axes = fig.add_subplot(gs[1, 1])
        #axes.hist([median_cadence[heats_mask]*24, median_cadence[~heats_mask]*24],
        axes.hist([group_cadence.loc[heats_tiles], group_cadence.loc[cold_tiles]],
                bins=np.arange(0, max(group_cadence.values)+1), stacked=True,
                color=['red', 'blue'],
                #label=['Tiles in the HEATS survey', 'Tiles in the COLD survey'],
        )
        #axes.legend()
        axes.set_xlabel('Days between observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(0-0.1, int(max(group_cadence.values))+2)
        axes.set_xticks(cadence_ticks)
        plt.savefig(f"{figure_dir}/reserve_fraction_{reserve_fraction/2:.2f}_summary.png", dpi=300)
        #plt.show()


In [ ]:
#tile_results = results_df[results_df['tile'] == 'T0500'].copy()
import matplotlib as mpl
tile_results = results_df.groupby(["tile","reserve_fraction", "pandas_time", ]).first()
cmap = colormaps['viridis_r']
colors = cmap(np.linspace(0.1, 01.0, len(reserve_fractions)))
fig, ax = plt.subplots()
for i, reserve_fraction in enumerate(reserve_fractions):
    series = tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean()
    ax.plot(series.index, series.values, color=colors[i], label=f'Reserve fraction {reserve_fraction}')
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])
cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.1,0.45), cmap='viridis_r'), ax=ax)
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())
plt.ylim(1.5, 35)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0444'
grid.plot(highlight=[tile])
plt.show()
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]
cmap = colormaps['viridis_r']
colors = cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig, ax = plt.subplots()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    # Mask values above 20 days for better visualization
    series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_title("Cadence by Month " + tile)

ax.yaxis.set_major_locator(plt.MultipleLocator(3))
ax.yaxis.set_minor_locator(plt.MultipleLocator(1))

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap='viridis_r'), ax=ax)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())
plt.ylim(0, 20)
#plt.legend(ncol=2)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0120'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]
cmap = colormaps['viridis_r']
colors = cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(10, 9), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 1], hspace=0.12)
gs.update(top=0.95,bottom=0.01, left=0.07, right=0.97)
ax = fig.add_subplot(gs[1, 0])
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 10)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap='viridis_r'), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.05, location='top', aspect=40)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())
ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='tab:red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='tab:blue', lw=0.7, zorder=1.4, alpha=0.3)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0100'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 10)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0, 3)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
#cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='tab:red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='tab:blue', lw=0.7, zorder=1.4, alpha=0.3)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0548'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 20)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0.25, 5.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='blue', lw=0.7, zorder=1.4, alpha=0.2)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0685'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 20)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0.25, 5.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='blue', lw=0.7, zorder=1.4, alpha=0.2)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0525'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 20)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0.25, 5.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='blue', lw=0.7, zorder=1.4, alpha=0.2)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0200'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 20)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0.25, 5.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='blue', lw=0.7, zorder=1.4, alpha=0.2)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)

In [ ]:
plt.rcParams['date.converter'] = 'concise'
tile = 'T0220'
plot_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

#make a custom colormap using plasma reversed but offset to start at 0.2
cmap = colormaps['plasma_r']
custom_colors = cmap([x+0.2 for x in plot_fractions])
print(custom_colors)
custom_cmap = mpl.colors.ListedColormap(custom_colors)
colors = custom_colors #cmap(plot_fractions)
#colors = colormaps['tab10'].colors
fig = plt.figure(figsize=(9, 13), dpi=100)
fig.suptitle(f'Cadence by Month: {tile}')
gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.1)
gs.update(top=0.96,bottom=0.03, left=0.07, right=0.97)

subgs = gs[1, 0].subgridspec(2, 1, height_ratios=[1, 1], hspace=0)

ax = fig.add_subplot(subgs[0, 0])
ax.xaxis.tick_top()
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].mean().copy()
    min = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].min().copy()
    max = tile_results.loc[tile, reserve_fraction].resample('SMS')['diff'].max().copy()
    # Mask values above 20 days for better visualization
    if tile in heats_tiles:
        series = series.where(series <= 20)
    else:
        series = series.where(series <= 20)
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}')
    ax.scatter(series.index, series.values, color=colors[i], s=10)
    ax.errorbar(series.index, series.values, yerr=[series - min, max - series], fmt='none', ecolor=colors[i], alpha=0.3, capsize=3)
    #tile_results.loc['T0100', reserve_fraction].resample('ME')['diff'].mean().plot(color=colors[i])

ax.set_xlim(np.datetime64('2026-01-01'), np.datetime64('2026-12-15'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())

if tile in cold_tiles:
    ax.set_ylim(0, 20)
    ax.yaxis.set_major_locator(plt.MultipleLocator(3))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
if tile in heats_tiles:
    ax.set_ylim(0.25, 5.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_ylabel("Cadence (Days)")
ax.grid(alpha=0.4)
ax.grid(which='minor', linestyle='dashed', alpha=0.2)

#cbar = fig.colorbar(mpl.cm.ScalarMappable(mpl.colors.Normalize(0.0,0.4), cmap='viridis_r'), ax=ax)
boundaries = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
norm = mpl.colors.BoundaryNorm(boundaries, custom_cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=custom_cmap), ax=ax, use_gridspec=True, orientation='horizontal', fraction=0.05, pad=0.1, location='top', aspect=50)
cbar.set_ticks([ x + 0.1/2 for x in boundaries[:-1]])
cbar.set_ticklabels([f"{x:.1f}" for x in boundaries[:-1]])
cbar.set_label('Reserve Fraction')
#ax.set_yscale('log')
#ax.yaxis.set_major_locator(plt.LogLocator(base=10.0, subs='all'))
#ax.yaxis.set_major_formatter(plt.ScalarFormatter())

# Plot number of observations per month
ax = fig.add_subplot(subgs[1, 0], sharex=ax)
for i, reserve_fraction in enumerate(plot_fractions):
    series = tile_results.loc[tile, reserve_fraction].resample('SMS').size().copy()
    ax.plot(series.index, series.values, color=colors[i], label=f'{reserve_fraction/2}', marker = 's', markersize=3)
ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
ax.set_ylabel("Number of Observations")
ax.grid(alpha=0.4)


ax = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
grid.plot_tiles(ax, tilenames=grid.tilenames, fc='none', ec='lightgrey', zorder=0.5)
grid.plot_tiles(ax, tilenames=heats_tiles, fc='tab:red', ec='red', lw=0.7, zorder=1.5, alpha=0.3)
grid.plot_tiles(ax, tilenames=cold_tiles, fc='tab:blue', ec='blue', lw=0.7, zorder=1.4, alpha=0.2)
if tile in heats_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='orange', ec='red', lw=2.0, zorder=1.6)
elif tile in cold_tiles:
    grid.plot_tiles(ax, tilenames=[tile], fc='tab:blue', ec='blue', lw=2.0, zorder=1.6)
plt.savefig(f"{figure_dir}/cadence_by_month_{tile}.png", dpi=300)